# C61.pdf 作者视角技术复盘：mmLock

论文：**mmLock: User Leaving Detection Against Data Theft via High-Quality mmWave Radar Imaging**

作者：Jiawei Xu, Ziqian Bi, Amit Singha, Tao Li, Yimin Chen, Yanchao Zhang  
会议：2023 32nd International Conference on Computer Communications and Networks (ICCCN)  
DOI：10.1109/ICCCN58024.2023.10230151

这份 notebook 用中文从作者视角复盘这项工作，不做逐字翻译，重点说明：

- 我们要解决什么安全问题；
- 为什么选择 FMCW mmWave radar；
- raw radar signal 如何变成 point cloud；
- 为什么需要三次 FFT：Range FFT、Doppler FFT、Angle FFT；
- 为什么使用 PointNet + LSTM；
- 实验如何设计，结果说明了什么；
- 这项工作后续如何整理成工程项目。

## 图片提取说明

PDF 中的图片已经提取到 `img/` 目录：

- `img/embedded/`：从 PDF 中直接抽取的内嵌图片对象，共 24 个。
- `img/pages/`：把 10 页 PDF 渲染成整页 PNG，防止矢量图或组合图漏掉。
- `img/embedded/`：后续 notebook 直接引用这里的原始抽取图片，避免手工裁剪导致位置偏移。

后面的解读优先使用 `img/embedded/` 中的原始图片。个别论文图是 PDF 矢量对象，不会出现在 `embedded/` 中，这类图直接用文字解释。

## 1. 一句话理解这项工作

我们在这项工作中提出 **mmLock**：用 TI IWR6843ISK-ODS 毫米波雷达采集用户身体的 3D 点云，再用 **PointNet + LSTM** 识别用户是否正在离开设备，从而在用户刚离开时立刻锁定设备，减少手机、平板、笔记本等设备被偷或遗失后产生的数据泄露风险。

它的核心不是“识别某个无线信号模式”，而是先把 mmWave 信号变成可以解释的 **人体点云/人体 mesh**，再从点云序列中识别 leaving gesture。这样比 RSSI、CSI 或 acoustic ranging 更直观，也更适合动态环境和多移动目标场景。

## 2. 文章结构总览

| 章节 | 内容 | 阅读重点 |
| --- | --- | --- |
| Abstract | 问题、方案、结果 | mmLock 的核心贡献 |
| I. Introduction | 安全动机与已有方案不足 | 为什么需要 user leaving detection |
| II. System Overview | 系统 pipeline | IF signal -> range heatmap -> point cloud -> PointNet-LSTM |
| III. Threat Model | 威胁模型 | 假设攻击者拿到未锁设备，但不讨论破解密码 |
| IV. Point Cloud Generation | 雷达点云生成 | 三次 FFT 和 TDM-MIMO 是核心 |
| V. Data Preprocessing | 数据预处理 | 过滤低质量 frame，DBSCAN 去噪和追踪用户点云 |
| VI. Leaving Gesture Detection | 模型设计 | PointNet 提空间特征，LSTM 提时间特征 |
| VII. Evaluation | 实现与评估 | 设备、数据规模、训练设置、检测性能 |
| VIII. Related Work | 相关工作 | one-time auth、continuous auth、acoustic、RSSI/CSI、mmWave imaging |
| IX. Conclusion | 总结 | mmWave point cloud 用于数据防盗 |


## 3. 系统图：mmLock 的主流程

<p>
  <img src="img/embedded/pdf_image-002-000.png" width="180" />
  <img src="img/embedded/pdf_image-002-002.png" width="220" />
  <img src="img/embedded/pdf_image-002-003.png" width="220" />
  <img src="img/embedded/pdf_image-002-004.png" width="160" />
</p>

这张图是我们整项工作的主线。整个系统从左到右可以拆成六步：

1. **IF Signals**：雷达发射 FMCW chirp，接收人体反射信号，混频后得到中频 IF 信号。
2. **Range Heatmap Preprocessing**：先看 range heatmap，过滤掉信号弱、不适合生成点云的 frame。
3. **Point Cloud Generation**：从雷达信号中估计 range、velocity、angle，生成 3D 点云。
4. **Point Cloud Preprocessing**：对点云做去噪、聚类、目标用户提取和点数规整。
5. **Training**：先用静态姿态预训练 PointNet，再用动态手势训练 LSTM。
6. **Prediction**：实际使用时，PointNet 提取每帧点云的空间特征，LSTM 根据时间序列判断是否离开。

最关键的工程思想是：**不要直接把原始雷达信号丢给黑盒模型，而是先构造可解释的 3D point cloud，再做行为识别。**

## 4. Abstract：我们要讲清楚的核心问题

摘要第一层意思：智能设备越来越多，设备丢失或被偷会带来隐私和数据安全问题。传统认证只能解决“谁能解锁”，但不能保证用户离开时设备马上锁定。

摘要第二层意思：已有 user leaving detection 主要依赖声学测距。声学方案的问题是：需要视距路径，容易被遮挡；而且当周围有多个移动物体时，难以区分真正离开的用户和其他移动目标。

摘要第三层意思：我们的 mmLock 用 mmWave FMCW radar 捕获用户 3D mesh，再用 PointNet-LSTM 从 3D human mesh 序列中检测 leaving gesture。点云是可解释的，不只是 raw signal pattern，所以鲁棒性更好。

摘要最后给出结果：我们基于商用 TI mmWave radar 实现系统，在多个环境和场景中评估，使用超过 1 TB 的 mmWave 信号数据训练，在多数场景中达到 100% true-positive rate。

## 5. Introduction：问题为什么重要

Introduction 先从现实问题开始：智能手机、平板、笔记本大量普及，设备里有通信、工作、娱乐、医疗等数据。一旦设备丢失或被偷，攻击者可能在设备还没锁屏前访问隐私数据。

我们强调的不是传统“破解密码”问题，而是一个更直接的时间窗口问题：用户离开后，设备可能还处于 unlocked 状态。即使设备设置了密码，自动锁屏时间也可能是 30 秒、几分钟，甚至 never。这个窗口足够攻击者查看数据。

因此，我们关注的安全目标是：**当合法用户开始离开设备时，系统要尽快识别并锁定设备。** 这比事后认证攻击者更主动。

## 6. Introduction：已有方案为什么不够

我们把已有防护思路分成几类来定位 mmLock 的价值：

1. **一次性认证**：密码、PIN、指纹、人脸等。问题是用户离开后设备仍可能处于解锁状态。
2. **连续认证**：通过触摸、打字、传感器等判断当前使用者是否为合法用户。问题是它通常要等攻击者使用一段时间后才能发现，无法阻止攻击者先看屏幕或复制数据。
3. **用户离开检测**：用户刚离开就锁定设备。我们认为这个方向更适合防止离开后的数据泄露。

已有 immediate locking 类工作很多依赖 acoustic ranging。声学方案的问题在于：

- 需要设备和用户之间有 LOS 路径；
- 容易被桌椅、身体、环境物体遮挡；
- 手机/电脑麦克风数量有限，难以在多人或多移动物体环境下区分目标用户。

## 7. Introduction：为什么 mmWave radar 适合这个问题

mmWave 的优势来自高频和短波长。频率越高、波长越短，空间分辨率越好。我们使用 60 GHz 到 64 GHz 的 FMCW radar，波长约 4 mm，适合近距离人体感知。

与 RSSI/CSI 这类通信信号特征相比，mmWave radar 可以更直接地估计人体反射点的位置，形成 3D point cloud。这样有三个好处：

- **可解释**：点云里能看到人体大致位置和动作，而不是难以解释的信号模式。
- **空间信息强**：可以利用 range、velocity、angle 来区分用户与环境。
- **动态环境更稳**：通过聚类和追踪，可以把合法用户与附近攻击者分开。

## 8. Threat Model：威胁模型

我们的威胁模型很明确：攻击者可能捡到或偷到设备，并尝试在设备未锁定时访问受害者隐私数据。

mmLock 的目标是：**在设备落入攻击者手里之前，确认设备已经因为用户离开而锁定。**

这项工作不试图解决：

- 攻击者长期暴力破解密码；
- 攻击者刷机、重装系统；
- 攻击者绕过操作系统安全机制。

所以我们的边界是“离开触发的即时锁定”，不是完整的设备安全体系。这个边界写清楚后，系统目标会更清晰，也更容易评估。

## 9. Point Cloud Generation：从雷达信号到点云

Figure 2 是 PDF 中的矢量示意图，`pdfimages` 不会把它作为独立位图抽出来。这里直接解释它表达的核心关系：FMCW chirp、物理 TX/RX 天线，以及 TDM-MIMO 形成的 virtual antenna array。

点云生成是这项工作最重要的技术部分。我们使用 TI IWR6843ISK-ODS：

- 3 个 TX antenna；
- 4 个 RX antenna；
- FMCW 频率从 60 GHz sweep 到 64 GHz；
- 使用 TDM-MIMO，把 3 TX × 4 RX 组合成 12 个 virtual antennas。

雷达发射 chirp，人体不同部位反射信号。接收信号与发射信号混频后得到 IF signal。IF signal 里混在一起的信息包括：

- 目标距离；
- 目标速度；
- 反射角度；
- 不同人体部位和环境物体的混合反射。

所以后面需要通过 FFT 把这些维度分开。

## 10. 为什么需要三次 FFT：总解释

FMCW 雷达接收到的原始数据不是点云，而是多维复数采样。一个反射点会在三个维度上留下可测特征：

| FFT | 操作维度 | 分离出的物理量 | 回答的问题 |
| --- | --- | --- | --- |
| Range FFT | 每个 chirp 内的 ADC sample，也叫 fast-time | 距离 range | 目标离雷达多远？ |
| Doppler FFT | 连续 chirp/loop，也叫 slow-time | 速度 velocity | 目标是在动还是静止？速度是多少？ |
| Angle FFT | RX/TX virtual antenna array，也叫 spatial dimension | 到达角 angle | 目标从哪个方向来？ |

三次 FFT 不是重复计算，而是在三个不同轴上做频域分解：

1. **时间采样轴**：从 beat frequency 得到 range。
2. **chirp 序列轴**：从相位随时间变化得到 Doppler velocity。
3. **天线阵列轴**：从不同天线接收相位差得到 angle。

只有 range，只能知道“某个距离上有反射”；只有 range + Doppler，可以知道“某个距离上有移动物体”；再加 angle，才能把反射点定位到 3D 空间，形成点云。

## 11. 第一次 FFT：Range FFT 为什么能得到距离

FMCW chirp 的频率随时间线性上升。信号从雷达到目标再反射回来，会有一个 round-trip delay。由于发射信号频率正在变化，延迟信号和当前发射信号之间会产生频率差，这个差就是 **beat frequency**。

距离越远，往返延迟越大；延迟越大，beat frequency 越大。因此：

- ADC sample 记录的是一个 chirp 内的 IF signal；
- 不同距离的反射，对应不同 beat frequency；
- 对 ADC sample 做 FFT，就能把不同 beat frequency 分开；
- beat frequency 再根据 chirp slope 换算成 range。

直观理解：Range FFT 把“一段时间里的混合波形”拆成“不同距离上的能量”。这就是 range heatmap 的基础。

## 12. 第二次 FFT：Doppler FFT 为什么能得到速度

如果目标在动，它和雷达之间的距离会连续变化。即使相邻 chirp 的距离变化很小，反射信号的相位也会发生可测变化。

Doppler FFT 的输入不是单个 chirp，而是一组连续 chirps。它观察同一个 range bin 在多个 chirp 上的相位变化：

- 静止物体：相位基本稳定，Doppler 接近 0；
- 移动物体：相位随 chirp 序列变化，产生非零 Doppler frequency；
- 相位变化越快，速度越大。

因此 Doppler FFT 的作用有两层：

1. 估计目标速度；
2. 帮助抑制桌子、椅子、墙面等静态 clutter。

对 user leaving detection 来说，Doppler 很关键，因为 leaving gesture 本质上是动态行为。只看 range，容易把静态环境误认为目标；加入 Doppler 后，系统更关注正在移动的人体反射。

## 13. 第三次 FFT：Angle FFT 为什么能得到方向

同一个反射点到达不同天线的路径长度略有不同，因此不同 RX antenna 接收到的相位也不同。这个相位差与信号到达角有关。

Angle FFT 的核心思想是：

- 把多个天线看成 spatial samples；
- 不同到达角对应不同的空间相位梯度；
- 对 antenna dimension 做 FFT，就能把不同角度方向上的能量分开。

IWR6843 物理天线数量有限，所以我们使用 TDM-MIMO：三个 TX 分时发射，每个 TX 的 chirp 都被四个 RX 接收。这样组合后得到 12 个 virtual antennas，提高角度分辨率。

Angle FFT 的结果用于估计 azimuth 和 elevation。结合 range，就能把反射点转换为 3D 坐标：`x, y, z`。这一步是从 radar cube 走向 point cloud 的关键。

## 14. 三次 FFT 后的数据形态

工程上可以把数据维度理解为下面这条链：

```text
raw ADC bin
  -> [frame, loop, tx, rx, adc_sample]
  -> Range FFT:   [frame, loop, tx, rx, range_bin]
  -> Doppler FFT: [frame, doppler_bin, tx, rx, range_bin]
  -> Angle FFT:   [frame, doppler_bin, angle_bin, range_bin]
  -> point cloud: [x, y, z, power, velocity]
```

这也解释了为什么项目里的 `radar_fft_cube_progress_zh.ipynb` 和并行处理代码要拆成三层 FFT：它们对应的是雷达物理模型里的三个独立维度，不是随便写的三个处理步骤。

## 15. MIMO：为什么要虚拟天线阵列

这里的关键判断是：仅靠物理接收天线时角度分辨率不够。角度分辨率和天线数量、天线间距有关。IWR6843 的物理天线数量有限，如果只用物理 RX，人体点云会比较粗糙。

TDM-MIMO 的做法是：

- 一个 loop 里安排三个 time slots；
- 三个 TX antenna 轮流发射 chirp；
- 每个 TX 发射时，4 个 RX 同时接收；
- 最终等效出 3 × 4 = 12 个 virtual antennas。

这样可以在不增加硬件天线数量的情况下提高角度估计能力。实际处理时，某些蓝色标记的虚拟天线有 π phase inversion，因此计算角度前需要做相位校正。这一点对复现很重要。

## 16. 原始点云与去噪

![Raw point cloud](img/embedded/pdf_image-002-003.png)

Figure 3 展示的是一个离开用户的原始点云。可以看到人体轮廓的大致结构，例如头、手臂、腿，也能看到环境噪声点。

这里有两个重要观察：

1. 人体躯干反射通常比手部或环境随机反射更强。
2. 原始点云虽然能看出人形，但仍然很稀疏、很 noisy，不能直接喂给行为识别模型。

所以后面必须做 range heatmap filtering 和 point cloud preprocessing。

## 17. Range heatmap 预处理

![Range heatmap](img/embedded/pdf_image-005-006.png)

Figure 4 是静态用户的 range heatmap。横轴是时间，纵轴是距离，颜色表示信号强度。

我们在数据中观察到：有些 frame 的用户反射很强，有些 frame 信号很弱，弱信号 frame 不适合生成点云。如果把低质量 frame 也放进模型，会让后续 PointNet-LSTM 学到噪声。

处理方法很直接：对每个 frame 的所有 range bins 信号强度求和，低于阈值的 frame 过滤掉。在我们的实验中，只保留约 33% 的 frame 用于点云生成和离开检测。

这个设计说明 mmLock 不追求“所有数据都用”，而是优先保证用于模型的数据质量。

## 18. 点云预处理：DBSCAN 聚类与点数规整

![Point cluster example](img/embedded/pdf_image-002-004.png)

从 Doppler heatmap 中选择显著像素后，我们用 TDM-MIMO 生成 3D 点。一个 Doppler heatmap 中如果选择 300 个 pixel，每个 pixel 可以生成 4 个点，那么单帧得到约 1200 个点。

由于单帧点云稀疏，我们把 5 个相邻 frame 合并，得到约 6000 个点。然后做两步去噪：

1. 用 DBSCAN 根据空间距离把点分成不同 cluster；
2. 根据 cluster 的总能量过滤弱 cluster，只保留目标用户对应的 cluster。

最后，PointNet 要求固定输入大小。我们把每个点云规整成 2048 个点：点太多就随机下采样，点太少就用层次聚类方式上采样。

## 19. 为什么使用 PointNet + LSTM

Figure 6 是网络结构矢量图，核心结构可以用文字理解：每一帧点云先经过 PointNet 得到空间特征，连续 K 帧空间特征再进入两层 LSTM，最后经过全连接层和 dropout 输出分类结果。

mmLock 的输入不是单张点云，而是一段时间内的点云序列。我们把问题拆成两层：

1. **单帧点云是什么姿态？** 用 PointNet 提取空间特征。
2. **一串姿态变化是不是离开动作？** 用 LSTM 提取时间特征。

PointNet 适合点云，因为点云没有天然顺序。同一个人体点云，即使点的排列顺序变了，或者发生旋转、平移，PointNet 仍然能提取相对稳定的空间特征。

LSTM 适合动作，因为 leaving gesture 是一个过程：坐着/站着 -> 起身 -> 转身 -> 远离。单帧只能看到姿态，序列才能看到动作趋势。

## 20. 有攻击者时如何追踪目标用户

![Attacker clusters](img/embedded/pdf_image-006-008.png)

我们不仅考虑单用户，还考虑附近有攻击者或其他移动物体。Figure 7 中，合法用户和攻击者会形成不同点云 cluster。

难点不是“能不能看到两个 cluster”，而是“跨 frame 判断哪个 cluster 属于合法用户”。mmLock 用两个指标做关联：

- cluster center：所有点坐标的平均位置；
- center of mass：按点的信号强度加权后的中心。

相邻 frame 时间很短，人体移动距离有限，因此合法用户 cluster 的 center 和 center of mass 不会突然跳很远。系统用这个连续性追踪目标用户。

如果攻击者离合法用户很近，两个 cluster 可能被 DBSCAN 合并。我们的处理思路是：观察 cluster 尺寸突然变大，并结合前几帧坐标把两个目标重新分开。

## 21. 系统实现：硬件和采集参数

![Testbed setup](img/embedded/pdf_image-006-009.png)

我们使用的硬件是：

- TI IWR6843ISK-ODS mmWave radar；
- TI DCA1000EVM 用于实时数据采集和 streaming；
- 雷达工作频段：60 GHz 到 64 GHz；
- 波长约 4 mm；
- 3 个 TX antenna，4 个 RX antenna；
- E-plane 和 H-plane FoV 都约 120°。

采集配置：

- 训练阶段：63 frames/s，24 chirp loops/frame；
- 测试阶段：33 frames/s，18 chirp loops/frame；
- 每个 chirp 578 samples；
- 最大检测距离约 15 m；
- 距离分辨率约 3.9 cm；
- 最大检测速度约 2 m/s；
- 速度分辨率约 0.27 m/s。

这些参数解释了为什么 notebook 和并行代码里必须把 `num_adc_samples`、`num_loops_per_frame`、`num_tx`、`num_rx`、`sample_rate`、`slope` 等做成配置项。参数不一致，FFT 后的物理量就会错。

## 22. 模型训练：PointNet 预训练

Table I 是 PDF 中的表格对象，这里不再使用裁剪图，直接保留关键设置：34 种静态姿态，其中至少 14 种与离开动作相关。

PointNet 先用静态姿态预训练。我们设计了 34 种静态姿态，其中至少 14 种与离开动作相关。

数据设置：

- 参与者：16 名大学生，11 男 5 女；
- 每种静态姿态保持 24 秒；
- 每人每个姿态采集 1500 frames；
- 每人每个姿态生成约 100 个 point cloud；
- PointNet 输入大小：2048 × 3；
- 输出类别：34 种姿态。

这一步的作用是让 PointNet 学会从点云中识别人体姿态，而不是直接端到端学习“离开/不离开”。这样后续 LSTM 得到的是更稳定、更有语义的 spatial feature。

## 23. 训练曲线：PointNet 和 LSTM

<p>
  <img src="img/embedded/pdf_image-007-011.png" width="360" />
  <img src="img/embedded/pdf_image-007-012.png" width="360" />
</p>

PointNet 训练中，loss 逐渐下降，accuracy 上升。PointNet 在约 100 iterations 后达到 95% 分类准确率。这里说明静态姿态特征是可以从 mmWave point cloud 中学习出来的。

<p>
  <img src="img/embedded/pdf_image-007-013.png" width="360" />
  <img src="img/embedded/pdf_image-007-014.png" width="360" />
</p>

LSTM 训练使用 PointNet 的倒数第二层空间特征作为输入。每个动态 gesture 包含 30 个 point clouds。LSTM 模型包括两层 100 units 的 LSTM，再接全连接层和 dropout。LSTM 更快收敛，并达到 100% 分类准确率。

这说明 pipeline 的设计是分层有效的：PointNet 学空间姿态，LSTM 学时间变化。

## 24. 单用户实验结果

我们先在单用户场景测试 false negative 和 false positive。

False negative 测试：

- 6 名学生；
- 默认环境；
- 坐着或站着面对雷达，然后按平常方式离开；
- 共 366 个 leaving samples；
- 系统全部检测到，true-positive rate = 100%。

False positive 测试：

- 用户做非离开动作，如前后摆动、左右摆动、站起、坐下、轻微运动；
- 系统误判 4 个动作为 leaving；
- false-positive rate = 1.67%。

这个结果说明系统对真正离开很敏感，同时对常见非离开动作误报较低。

## 25. 离开角度、位置、速度、高度和环境

Figure 11 是离开角度的矢量示意图：用户从 0° 到 180° 的五个方向离开，覆盖正前、斜前和侧向离开情况。

我们测试了 0° 到 180° 的 5 个离开角度，每个角度 20 次，全部正确识别。说明模型不是只学会某一个固定方向的离开动作。

Figure 12 是初始位置示意图：用户从三个不同初始位置离开，验证模型不是只依赖固定距离或固定站位。

我们还测试了 3 个初始离开位置，共 60 次，也全部正确识别。除此之外，还测试了：

- 初始距离：1 m、1.6 m、2.1 m、3.8 m；
- 离开速度：慢速、正常、快速，约 1.1、1.5、2.0 steps/s；
- 雷达高度：1 m、1.2 m、1.4 m、1.6 m；
- 环境：教室、图书馆 lobby、study area。

这些实验共同说明：mmLock 对常见位置变化、速度变化、设备高度变化和环境变化有一定鲁棒性。

## 26. 附近攻击者实验

Figure 13 是附近攻击者实验的矢量示意图：合法用户在中心位置，攻击者出现在四个不同方位并执行离开动作。

攻击者实验的设置是：合法用户在蓝色方块处，攻击者出现在 1 到 4 号位置。每个位置下，合法用户和攻击者都各做 20 次 leaving gestures。另外，攻击者还在位置 4 和 2 之间来回走动。

Figure 14 是 precision/recall 柱状图。由于它是矢量图，这里直接解释结果：四个攻击者位置下 recall 均为 100%，precision 均高于 90%。

结果：

- 所有位置的 recall 都是 100%，说明合法用户离开没有漏检；
- precision 都高于 90%；
- 位置 2 和 3 的 precision 分别约为 91% 和 95%；
- 有 3 次攻击者离开被错误关联为合法用户离开，原因是不同 frame 中的 point cluster association 出错；
- 攻击者在位置 2 和 4 之间来回走动没有触发 false positives。

这部分展示了 mmLock 与声学方案相比的重要优势：它不是只看“有人移动”，而是尝试在点云层面持续追踪目标用户。

## 27. Related Work 解读

相关工作部分主要对比四类技术：

1. **一次性认证**：密码、PIN、生物特征、随身硬件。问题是不能处理用户忘记锁屏后设备被拿走。
2. **连续认证**：用传感器、触摸、打字等持续验证使用者。问题是检测有延迟，攻击者可能先看到内容。
3. **声学离开检测**：可以做即时锁定，但依赖 LOS，且难以区分多移动目标。
4. **无线手势识别和 mmWave imaging**：RSSI/CSI 受方向、环境、用户影响大；已有 mmWave 点云工作未解决多移动物体下持续识别目标用户的问题。

mmLock 的定位是：用 COTS mmWave radar 生成更高质量、可解释的人体点云，并把它用于设备安全场景中的 user leaving detection。

## 28. Conclusion 解读

最后总结一下：mmLock 是一个基于 TI mmWave radar 的 user leaving detection 系统，用于防止设备离开场景下的数据盗取。

它的贡献可以概括为三点：

1. **安全问题定义清楚**：目标是用户离开时立即锁定设备，而不是事后识别攻击者。
2. **感知表示更强**：把雷达信号转换成高质量 point cloud，不依赖难解释的 raw signal pattern。
3. **模型结构合理**：PointNet 处理单帧点云空间特征，LSTM 处理多帧时间动作特征。

实验显示系统在多种离开姿态、角度、速度、位置、环境和附近攻击者场景中都有较好表现。

## 29. 与工程代码的对应关系

本项目里已经有两个工程材料可以和这项工作对应起来：

- `radar_fft_cube_progress_zh.ipynb`：解释从 DCA1000 raw ADC bin 到三次 FFT、radar cube 和 point cloud 的单机流程。
- `radar_fft_cube_progress_parallel/`：把读取、三层 FFT、点云提取拆成模块，并用 multiprocessing pool 做 frame-level 并行处理。

Section IV 对应代码里的核心模块：

| 文章概念 | 工程模块 |
| --- | --- |
| raw IF/ADC signal | DCA1000 bin reader |
| Range FFT | `range_fft()` |
| Doppler FFT | `doppler_fft()` |
| Angle FFT | `angle_fft()` |
| point cloud generation | point extraction / coordinate conversion |
| point cloud preprocessing | DBSCAN / filtering / fixed-size point cloud |
| PointNet-LSTM | 后续 deep learning 训练模块 |
